# 06. Amazon Bedrock Intelligent Prompt Routing 체험

지금까지의 실습(SGLang decision layer)이 **"판단용 LLM 호출을 아예 제거"** 하는 접근이라면, Bedrock **Intelligent Prompt Routing(IPR)** 은 **"생성 호출이 어느 모델로 갈지를 요청마다 골라주는"** 관리형 기능입니다.

| 비교 | Decision layer (01-05 실습) | Intelligent Prompt Routing |
|---|---|---|
| 하는 일 | 선택지 중 label 하나를 즉답 (prefill-only) | 요청 난이도를 예측해 같은 family의 작은/큰 모델 중 선택 |
| 대상 | agent 내부의 판단 노드 | 고객 응답을 생성하는 LLM 호출 자체 |
| 효과 | 판단 latency 수십 ms, 판단 비용 ~0 | 쉬운 요청을 작은 모델로 보내 생성 비용 절감 |
| 관계 | **보완재** — 판단은 decision layer로 제거하고, 남은 생성 호출은 IPR로 라우팅할 수 있습니다 | |

호출 방법은 간단합니다: `converse()`의 `modelId` 자리에 **router ARN**을 넣으면, 응답의 `trace.promptRouter.invokedModelId`로 어느 모델이 선택됐는지 알 수 있습니다.

## 1. 이 리전의 prompt router 목록

us-west-2에는 family별 **default router** 3종이 제공되고, `CreatePromptRouter`로 **custom router**도 만들 수 있습니다.

> **실습 모델 안내 (2026-10 기준)**: default Anthropic router는 Claude 3 Haiku / 3.5 Sonnet을 멤버로 참조하는데 이 모델들은 **EOL**이어서 호출이 거부되고, 최신 Claude 4.5 계열은 아직 prompt routing **미지원**입니다(CreatePromptRouter가 ValidationException 반환 — 직접 확인). 따라서 이 노트북은 **Nova family(Lite/Pro)** 로 라이브 실습합니다. 동작 원리와 API는 family와 무관하게 동일합니다.

In [1]:
import boto3, json, time, statistics
from IPython.display import Markdown, display

REGION = "us-west-2"
br = boto3.client("bedrock", region_name=REGION)
rt = boto3.client("bedrock-runtime", region_name=REGION)

rows = ["| Router | type | qualityDiff | 멤버 모델 |", "|---|---|---|---|"]
for r in br.list_prompt_routers()["promptRouterSummaries"]:
    models = ", ".join(m["modelArn"].split("/")[-1] for m in r["models"])
    rows.append(f"| {r['promptRouterName']} | {r['type']} | "
                f"{r['routingCriteria']['responseQualityDifference']} | {models} |")
display(Markdown("\n".join(rows)))

| Router | type | qualityDiff | 멤버 모델 |
|---|---|---|---|
| Nova Prompt Router | default | 0.0 | us.amazon.nova-lite-v1:0, us.amazon.nova-pro-v1:0 |
| Anthropic Prompt Router | default | 0.0 | us.anthropic.claude-3-haiku-20240307-v1:0, us.anthropic.claude-3-5-sonnet-20240620-v1:0 |
| Meta Prompt Router | default | 0.0 | us.meta.llama3-1-8b-instruct-v1:0, us.meta.llama3-1-70b-instruct-v1:0 |

## 2. 같은 상담 질문을 router로 보내 보기

01-04 실습에서 썼던 한국어 상담 질문들을 **Nova default router**에 그대로 보내고, 요청마다 **어느 모델이 선택되는지**와 latency를 관찰합니다. 쉬운 인사/FAQ는 Lite로, 복잡한 송금 문의는 Pro로 가는지 확인해 보세요.

In [2]:
DEFAULT_NOVA = "arn:aws:bedrock:us-west-2:051934905721:default-prompt-router/amazon.nova:1"

QUESTIONS = [
    ("인사/FAQ (쉬움)", "안녕하세요? 영업시간이 어떻게 되나요?"),
    ("카드 문의 (중간)", "카드가 안 긁히는데 어떻게 해야 하나요? 간단히 답해줘."),
    ("송금 문의 (어려움)", "해외 송금이 이틀째 도착하지 않았습니다. SWIFT 중개은행 수수료 차감, "
                        "수취은행 컷오프, AML 보류 각각의 확인 절차와 고객 안내 문구를 정리해 주세요."),
]

def route_once(router_arn, text, max_tokens=200):
    t0 = time.perf_counter()
    r = rt.converse(modelId=router_arn,
                    messages=[{"role": "user", "content": [{"text": text}]}],
                    inferenceConfig={"maxTokens": max_tokens})
    ms = (time.perf_counter() - t0) * 1000
    picked = r.get("trace", {}).get("promptRouter", {}).get("invokedModelId", "?").split("/")[-1]
    answer = r["output"]["message"]["content"][0]["text"]
    return picked, ms, answer, r["usage"]

for tag, q in QUESTIONS:
    print(f"[{tag}]")
    print(f"  고객 질문: {q}")
    picked, ms, answer, usage = route_once(DEFAULT_NOVA, q)
    print(f"  선택 모델: {picked}   ({ms:,.0f}ms, in {usage['inputTokens']}/out {usage['outputTokens']} tok)")
    print(f"  답변: {answer.strip()[:160]}")
    print()


[인사/FAQ (쉬움)]
  고객 질문: 안녕하세요? 영업시간이 어떻게 되나요?


  선택 모델: us.amazon.nova-lite-v1:0   (1,384ms, in 26/out 200 tok)
  답변: 안녕하세요! 저는 챗봇이고, 제가 속한 서비스는 연중무휴 24시간 운영되고 있습니다. 언제든지 도움이 필요하시면 저에게 연락해 주세요. 만약 특정 업체나 서비스의 영업 시간에 대해 궁금하시다면, 해당 업체의 이름이나 위치를 알려주시면 더 구체적인 정보를 찾아 드릴 수 있습니다. 어떤 도

[카드 문의 (중간)]
  고객 질문: 카드가 안 긁히는데 어떻게 해야 하나요? 간단히 답해줘.


  선택 모델: us.amazon.nova-pro-v1:0   (1,351ms, in 40/out 200 tok)
  답변: 카드가 안 긁히는 문제는 여러 가지 이유로 발생할 수 있습니다. 다음과 같은 방법을 시도해 보세요:

1. **카드 확인**: 카드에 이상이 없는지 확인하세요. 휘어짐, 균열, 마그네틱 스트라이프 손상 등을 점검하세요.
2. **기기 확인**: 카드 리더기나 결제 단말기에 이상이 없는지

[송금 문의 (어려움)]
  고객 질문: 해외 송금이 이틀째 도착하지 않았습니다. SWIFT 중개은행 수수료 차감, 수취은행 컷오프, AML 보류 각각의 확인 절차와 고객 안내 문구를 정리해 주세요.


  선택 모델: us.amazon.nova-pro-v1:0   (2,205ms, in 100/out 200 tok)
  답변: 해외 송금이 도착하지 않을 때 고려해야 할 몇 가지 주요 절차와 고객에게 안내할 수 있는 문구를 정리해 보겠습니다.

### 1. SWIFT 중개은행 수수료 차감 확인
**절차:**
1. **송금 명세 확인:** 송금 명세서를 통해 중개은행이 수수료를 차감했는지 확인합니다.
2. **중



## 3. Custom router 만들기 — 라우팅 성향 조절

`routingCriteria.responseQualityDifference`(%, 5의 배수)는 "작은 모델의 예상 품질이 큰 모델보다 이만큼 낮아도 작은 모델을 쓰겠다"는 허용폭입니다. 값이 클수록 작은 모델로 더 공격적으로 라우팅됩니다.

아래 custom router(허용폭 25%)는 이 계정에 이미 생성되어 있습니다. 생성 코드는 다음과 같습니다(재실행 불필요):

```python
br.create_prompt_router(
    promptRouterName="ws-nova-router",
    routingCriteria={"responseQualityDifference": 25},   # %, 5의 배수만 허용
    models=[{"modelArn": ".../inference-profile/us.amazon.nova-lite-v1:0"},
            {"modelArn": ".../inference-profile/us.amazon.nova-pro-v1:0"}],
    fallbackModel={"modelArn": ".../inference-profile/us.amazon.nova-pro-v1:0"})
```

In [3]:
CUSTOM_NOVA = "arn:aws:bedrock:us-west-2:051934905721:prompt-router/59hrfjn7pynr"  # ws-nova-router

rows = ["| 질문 | default (diff=0) | custom (diff=25) |", "|---|---|---|"]
for tag, q in QUESTIONS:
    p1, m1, _, _ = route_once(DEFAULT_NOVA, q)
    p2, m2, _, _ = route_once(CUSTOM_NOVA, q)
    rows.append(f"| {tag} | {p1.replace('us.amazon.','')} ({m1:,.0f}ms) "
                f"| {p2.replace('us.amazon.','')} ({m2:,.0f}ms) |")
display(Markdown("\n".join(rows)))

| 질문 | default (diff=0) | custom (diff=25) |
|---|---|---|
| 인사/FAQ (쉬움) | nova-lite-v1:0 (1,881ms) | nova-lite-v1:0 (1,644ms) |
| 카드 문의 (중간) | nova-pro-v1:0 (1,372ms) | nova-lite-v1:0 (2,140ms) |
| 송금 문의 (어려움) | nova-pro-v1:0 (1,719ms) | nova-pro-v1:0 (1,923ms) |

## 4. Decision layer와 어떻게 함께 쓰나

- **판단(D1-D8)**: IPR로는 해결되지 않습니다 — 어느 모델로 라우팅하든 판단 1회는 LLM 생성 호출 1회(수백 ms, 과금)입니다. 판단은 decision layer(수십 ms, 비용 ~0)가 맞습니다.
- **생성(respond, summarize)**: 여기가 IPR의 자리입니다. 쉬운 상담 응답은 작은 모델이 쓰게 하여 생성 비용을 줄입니다.
- 조합 예: `guardrail/routing/escalation = SGLang decision` + `최종 응답 생성 = IPR(작은/큰 모델 자동 선택)` — 판단 latency와 생성 비용을 동시에 줄이는 구성입니다.

**유의점**
- router의 난이도 예측 자체도 완벽하지 않으므로, 품질 민감 업무는 responseQualityDifference를 보수적으로(0-10) 두고 시작하세요.
- IPR은 **같은 family 안에서만** 라우팅합니다 (Claude<->Nova 혼합 불가).
- latency는 선택된 모델의 생성 시간이 지배합니다 — IPR은 비용 최적화 도구이지, decision layer 같은 latency 제거 도구가 아닙니다.
- 과금: IPR 자체 요금 **$1 / 1,000 requests** + 선택된 모델의 토큰 요금 (aws.amazon.com/bedrock/pricing). 판단 1회당 $0.001 고정비가 추가되는 셈이라, 수십만 건 이상의 '판단' 용도로는 decision layer(한계비용 ~0)와 비용 구조가 근본적으로 다릅니다.